# Offline retrieval evaluation — `faiss_index_cases` (bge-base-en-v1.5)

Evaluates the **cases** retriever against `golden_queries.jsonl` (439 queries).
No API / LLM calls: the query is embedded with the same bge model that built the
index, FAISS returns the nearest chunks, chunks are grouped by case citation, and
retrieval metrics + an unanswerable-score gate are reported with bootstrap CIs.

Run all cells top-to-bottom. Outputs are written to `eval_results_cases.json`
and `eval_summary_cases.txt` in this folder.


In [1]:
# Offline retrieval evaluation for the bge-base FAISS index. No API calls.
import json, pickle, re, random, hashlib
import numpy as np
import faiss, torch
from sentence_transformers import SentenceTransformer

INDEX_DIR = "../faiss_index_cases"
GOLD_FILE = "golden_queries.jsonl"
MODEL_NAME = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
FETCH_K = 200        # chunks retrieved per query, before grouping by case
TOP_K = 10           # distinct cases kept per query
SEED = 42
N_BOOT = 2000
THRESH_GRID = np.round(np.arange(0.30, 0.901, 0.01), 2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SUMMARY_FILE = "eval_summary_cases.txt"
RESULTS_FILE = "eval_results_cases.json"

lines = []
def say(s=""):
    print(s, flush=True)
    lines.append(s)


In [2]:
# ---------- citation normalisation (used for index labels AND gold labels) ----------
def canon(s):
    s = re.sub(r"(?i)citation\s*name\s*:", " ", s or "")   # scraper prefix
    s = re.split(r"(?i)bookmark", s)[0]                    # scraper suffix
    return re.sub(r"[^a-z0-9]", "", s.lower())

# ---------- load index ----------
def load_index():
    index = faiss.read_index(f"{INDEX_DIR}/index.faiss")
    with open(f"{INDEX_DIR}/index.pkl", "rb") as f:
        docstore, id_map = pickle.load(f)   # needs the same langchain packages installed
    d = getattr(docstore, "_dict", docstore)
    chunk_keys = []
    for i in range(index.ntotal):
        doc = d[id_map[i]]
        meta = getattr(doc, "metadata", {}) or {}
        chunk_keys.append(canon(meta.get("citation", "")))
    return index, chunk_keys

# ---------- score conversion ----------
def make_to_cos(index):
    is_l2 = index.metric_type == faiss.METRIC_L2
    def to_cos(d):
        # normalised vectors: squared L2 = 2 - 2*cos, so cos = 1 - d/2
        return 1.0 - d / 2.0 if is_l2 else float(d)
    return to_cos, ("L2" if is_l2 else "inner product")

# ---------- search ----------
def search_all(queries, index, chunk_keys, model, to_cos):
    Q = model.encode([QUERY_PREFIX + q for q in queries],
                     normalize_embeddings=True, batch_size=32,
                     convert_to_numpy=True, show_progress_bar=True).astype("float32")
    D, I = index.search(Q, FETCH_K)
    ranked_all = []
    for drow, irow in zip(D, I):
        best = {}
        for d, i in zip(drow, irow):
            if i < 0:
                continue
            k = chunk_keys[int(i)]
            if not k:
                continue
            s = to_cos(float(d))
            if k not in best or s > best[k]:
                best[k] = s          # best chunk represents each case
        ranked = sorted(best.items(), key=lambda kv: -kv[1])[:TOP_K]
        ranked_all.append(ranked)
    return ranked_all

# ---------- metrics ----------
def retrieval_metrics(recs):
    ranks = [r["rank"] for r in recs]
    def hit(K):
        return [1.0 if (r is not None and r <= K) else 0.0 for r in ranks]
    return {
        "Recall@1":  hit(1),
        "Recall@5":  hit(5),
        "Recall@10": hit(10),
        "MRR@10":    [1.0 / r if (r is not None and r <= TOP_K) else 0.0 for r in ranks],
        "nDCG@10":   [1.0 / np.log2(r + 1) if (r is not None and r <= TOP_K) else 0.0
                      for r in ranks],
    }

def bootstrap_ci(vals, rng):
    v = np.asarray(vals, float)
    idx = rng.integers(0, len(v), size=(N_BOOT, len(v)))
    means = v[idx].mean(axis=1)
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def show_block(title, recs, rng):
    say(f"\n{title}  (n={len(recs)})")
    if not recs:
        say("  no queries")
        return
    for name, vals in retrieval_metrics(recs).items():
        mu, lo, hi = bootstrap_ci(vals, rng)
        say(f"  {name:10s} {mu:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

def auc(pos, neg):
    pos, neg = np.asarray(pos), np.asarray(neg)
    gt = (pos[:, None] > neg[None, :]).mean()
    eq = (pos[:, None] == neg[None, :]).mean()
    return float(gt + 0.5 * eq)

def gate_stats(ans_scores, un_scores, thr):
    tpr = float(np.mean(np.asarray(ans_scores) >= thr))   # answerable and answered
    tnr = float(np.mean(np.asarray(un_scores) < thr))     # unanswerable and refused
    return (tpr + tnr) / 2, tpr, tnr


In [3]:
# ---------- main ----------
def main():
    gold_hash = hashlib.sha256(open(GOLD_FILE, "rb").read()).hexdigest()
    rows = [json.loads(l) for l in open(GOLD_FILE, encoding="utf-8") if l.strip()]

    index, chunk_keys = load_index()
    to_cos, metric_name = make_to_cos(index)
    index_keys = set(chunk_keys) - {""}
    v0 = index.reconstruct(0)

    say("=== Setup ===")
    say(f"Index folder     : {INDEX_DIR}")
    say(f"Vectors          : {index.ntotal}   dim {index.d}   metric {metric_name}")
    say(f"Vector 0 norm    : {np.linalg.norm(v0):.4f}  (should be about 1.0)")
    say(f"Distinct cases   : {len(index_keys)}")
    say(f"Model            : {MODEL_NAME}  device {DEVICE}")
    say(f"FETCH_K / TOP_K  : {FETCH_K} / {TOP_K}")
    say(f"Golden file      : {GOLD_FILE}  sha256 {gold_hash[:16]}...  rows {len(rows)}")

    # coverage: are gold cases present in the index at all?
    gold_all = {canon(c) for r in rows if r["answerable"] for c in r["relevant_citations"]} - {""}
    gold_found = gold_all & index_keys
    say(f"Gold cases       : {len(gold_all)}   found in index: {len(gold_found)} "
        f"({100*len(gold_found)/max(len(gold_all),1):.1f}%)")
    if len(gold_found) < 0.9 * len(gold_all):
        say("WARNING: many gold citations are missing from the index. "
            "Check citation format before trusting the scores.")

    # search
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    ranked_all = search_all([r["query"] for r in rows], index, chunk_keys, model, to_cos)

    # per-query records
    results = []
    for row, ranked in zip(rows, ranked_all):
        keys = [k for k, _ in ranked]
        rec = {"id": row["id"], "query": row["query"], "query_type": row["query_type"],
               "answerable": row["answerable"],
               "top_score": float(ranked[0][1]) if ranked else -1.0,
               "top_cases": [{"citation_norm": k, "cos": round(s, 4)} for k, s in ranked[:5]]}
        if row["answerable"]:
            gold = {canon(c) for c in row["relevant_citations"]} - {""}
            rec["gold"] = sorted(gold)
            rec["gold_in_index"] = bool(gold & index_keys)
            rec["rank"] = next((i + 1 for i, k in enumerate(keys) if k in gold), None)
        results.append(rec)

    rng = np.random.default_rng(SEED)
    ans = [r for r in results if r["answerable"]]
    un = [r for r in results if not r["answerable"]]
    ans_in_index = [r for r in ans if r["gold_in_index"]]

    # retrieval results
    say("\n=== Retrieval: all answerable queries ===")
    show_block("All answerable", ans, rng)
    say("\n=== Retrieval: ceiling (gold case is in the index) ===")
    show_block("Answerable, gold in index", ans_in_index, rng)
    say("\n=== By query type (gold in index) ===")
    for qt in ("paraphrase", "direct"):
        show_block(qt, [r for r in ans_in_index if r["query_type"] == qt], rng)

    # unanswerable gate: threshold chosen on dev, reported on test
    grp_ans = sorted({tuple(r["gold"]) for r in ans})
    random.Random(SEED).shuffle(grp_ans)
    dev_groups = set(grp_ans[: len(grp_ans) // 2])
    un_shuffled = list(range(len(un)))
    random.Random(SEED + 1).shuffle(un_shuffled)
    dev_un_idx = set(un_shuffled[: len(un) // 2])

    dev_ans = [r["top_score"] for r in ans if tuple(r["gold"]) in dev_groups]
    test_ans = [r["top_score"] for r in ans if tuple(r["gold"]) not in dev_groups]
    dev_un = [un[i]["top_score"] for i in range(len(un)) if i in dev_un_idx]
    test_un = [un[i]["top_score"] for i in range(len(un)) if i not in dev_un_idx]

    thr = float(max(THRESH_GRID, key=lambda t: gate_stats(dev_ans, dev_un, t)[0]))
    bal, tpr, tnr = gate_stats(test_ans, test_un, thr)

    say("\n=== Unanswerable gate ===")
    say(f"Score separation, all queries: AUC = "
        f"{auc([r['top_score'] for r in ans], [r['top_score'] for r in un]):.3f}  "
        f"(0.5 = no separation, 1.0 = perfect)")
    say(f"Threshold chosen on dev half: {thr:.2f}")
    say(f"Test half: answered {tpr:.3f} of answerable (n={len(test_ans)}), "
        f"refused {tnr:.3f} of unanswerable (n={len(test_un)}), balanced {bal:.3f}")
    say(f"Top-score percentiles (10/50/90)  answerable:   "
        f"{np.percentile([r['top_score'] for r in ans], [10,50,90]).round(3)}")
    say(f"Top-score percentiles (10/50/90)  unanswerable: "
        f"{np.percentile([r['top_score'] for r in un], [10,50,90]).round(3)}")

    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=1)
    with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    say(f"\nSaved: {RESULTS_FILE}, {SUMMARY_FILE}")


In [4]:
main()


=== Setup ===
Index folder     : ../faiss_index_cases
Vectors          : 209168   dim 768   metric L2
Vector 0 norm    : 1.0000  (should be about 1.0)
Distinct cases   : 15627
Model            : BAAI/bge-base-en-v1.5  device cpu
FETCH_K / TOP_K  : 200 / 10
Golden file      : golden_queries.jsonl  sha256 b7392a91b65b2e95...  rows 439
Gold cases       : 200   found in index: 54 (27.0%)

=== Retrieval: all answerable queries ===

All answerable  (n=400)
  Recall@1   0.018   95% CI [0.005, 0.033]
  Recall@5   0.030   95% CI [0.015, 0.048]
  Recall@10  0.045   95% CI [0.028, 0.068]
  MRR@10     0.025   95% CI [0.012, 0.039]
  nDCG@10    0.029   95% CI [0.015, 0.044]

=== Retrieval: ceiling (gold case is in the index) ===

Answerable, gold in index  (n=108)
  Recall@1   0.065   95% CI [0.028, 0.111]
  Recall@5   0.111   95% CI [0.056, 0.176]
  Recall@10  0.167   95% CI [0.102, 0.241]
  MRR@10     0.091   95% CI [0.047, 0.143]
  nDCG@10    0.108   95% CI [0.061, 0.163]

=== By query type (gol